Dependências
- Instala o kagglehub, caso necessário
- Carrega bibliotecas necessárias para execução

In [4]:
!pip install kagglehub[pandas-datasets]

In [5]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import torch
import os
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
import torch.nn as nn
import copy
import torch.optim as optim
from sklearn.metrics import classification_report, confusion_matrix

device = "cuda" if torch.cuda.is_available() else "cpu"

Carregar Dataset
- Primeiramente, baixa o dataset na maquina pelo kagglehub

In [6]:
file_path = "HAM10000_metadata.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "kmader/skin-cancer-mnist-ham10000",
  file_path,
)
root = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

print(df.head())

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
     lesion_id      image_id   dx dx_type   age   sex localization
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear


Criação de Classe para facilitar transformações no dataset

In [7]:
# Tratamento Dataset
labels = dict()

for i, l in  enumerate(df['dx'].unique()):
    labels[l] = i

class HAMDataset(Dataset):
    def __init__(self, dataframe, root, transform=None):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.root = root
        self.transform = transform

        self.img_paths = []
        self.img_labels = []

        for i in range(len(self.dataframe)):
            row = self.dataframe.iloc[i]
            img_name = row['image_id'] + '.jpg'
            diagnosis = row['dx']

            # Verifica se esta na part_1 ou part_2
            img_path = os.path.join(root, 'HAM10000_images_part_1', img_name)
            if not os.path.exists(img_path):
                img_path = os.path.join(root, 'HAM10000_images_part_2', img_name)

            self.img_paths.append(img_path)
            self.img_labels.append(labels[diagnosis])

        print(f"Dataset preparado: {len(self.img_paths)} imagens válidas de {len(dataframe)} totais")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.item()

        img_path = self.img_paths[idx]
        label = self.img_labels[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label


Criação das transformações a serem aplicadas nas imagens (Vetorização, Redimensionalização, Normalização...)

In [8]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['dx'],
    random_state=42
)

print(f"Treino: {len(train_df)} imagens | Teste: {len(test_df)} imagens")
print(f"Distribuição de classes no treino:\n{train_df['dx'].value_counts()}\n")

train_dataset = HAMDataset(dataframe=train_df, root=root, transform=train_transform)
test_dataset  = HAMDataset(dataframe=test_df,  root=root, transform=test_transform)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


Treino: 8012 imagens | Teste: 2003 imagens
Distribuição de classes no treino:
dx
nv       5364
mel       890
bkl       879
bcc       411
akiec     262
vasc      114
df         92
Name: count, dtype: int64

Dataset preparado: 8012 imagens válidas de 8012 totais
Dataset preparado: 2003 imagens válidas de 2003 totais


In [9]:
# Função treino generalizada para todos os modelos
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(dataloader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        if batch_idx % 50 == 0:
            print(f'Batch {batch_idx}/{len(dataloader)} - Loss: {loss.item():.2f}')

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

Aplicação de vários modelos de treino, para avaliação do melhor.
Em conjunto, seleção dos parametros para cada determinado modelo.

In [10]:
# Configurações Especiais pra teste
# Lista de modelos para testar
lst_models_config = [
    # Modelos classicos
    {'name': 'VGG16', 'class': models.vgg16, 'pretrained': True},
    {'name': 'ResNet18', 'class': models.resnet18, 'pretrained': True},
    {'name': 'ResNet50', 'class': models.resnet50, 'pretrained': True},

    # EfficientNets
    {'name': 'EfficientNet-B0', 'class': models.efficientnet_b0, 'pretrained': True},
    {'name': 'EfficientNet-B2', 'class': models.efficientnet_b2, 'pretrained': True},

    # Transformers
    {'name': 'SwinTransformer', 'class': models.swin_t, 'pretrained': True},

    # Balanceados
    {'name': 'DenseNet121', 'class': models.densenet121, 'pretrained': True},
    {'name': 'MobileNetV3', 'class': models.mobilenet_v3_large, 'pretrained': True},

    # Especificos
    {'name': 'ConvNeXt', 'class': models.convnext_tiny, 'pretrained': True}
]

lst_num_epochs = [3, 5, 10]

def create_model(model_config, num_classes=7, freeze_backbone=True):
    model_name = model_config['name']
    model_class = model_config['class']
    pretrained = model_config['pretrained']

    # Carregar modelo
    model = model_class(pretrained=pretrained)

    # Congelar backbone se necessário
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Modificar última camada para 7 classes
    if 'vgg' in model_name.lower():
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            # Descongelar apenas classifier
            for param in model.classifier.parameters():
                param.requires_grad = True

    elif 'resnet' in model_name.lower():
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            for param in model.fc.parameters():
                param.requires_grad = True

    elif 'efficientnet' in model_name.lower():
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            for param in model.classifier.parameters():
                param.requires_grad = True

    elif 'vit' in model_name.lower():
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            for param in model.heads.parameters():
                param.requires_grad = True

    elif 'swin' in model_name.lower():
        in_features = model.head.in_features
        model.head = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            for param in model.head.parameters():
                param.requires_grad = True

    elif 'densenet' in model_name.lower():
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            for param in model.classifier.parameters():
                param.requires_grad = True

    elif 'mobilenet' in model_name.lower():
        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            for param in model.classifier.parameters():
                param.requires_grad = True

    elif 'inception' in model_name.lower():
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, num_classes)
        if freeze_backbone:
            for param in model.fc.parameters():
                param.requires_grad = True
            for param in model.AuxLogits.parameters():
                param.requires_grad = True

    elif 'convnext' in model_name.lower():
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            for param in model.classifier.parameters():
                param.requires_grad = True

    else:
        raise ValueError(f"Modelo {model_name} não suportado")

    return model, model_name

In [11]:
# Criar diretório para salvar modelos
os.makedirs("models", exist_ok=True)

Execução dos treinos e avaliações dos modelos

In [ ]:
# Lista para armazenar todos os resultados
all_results = []

# Treinamento
for model_config in lst_models_config:
    model_name = model_config['name']
    print(f"Modelo: {model_name}")

    for num_epochs in lst_num_epochs:
        print(f"Treinando {model_name} por {num_epochs} epochs")

        # Novo modelo para experimento
        model, _ = create_model(model_config, num_classes=7, freeze_backbone=True)
        model = model.to(device)

        criterion = nn.CrossEntropyLoss()
        trainable_params = filter(lambda p: p.requires_grad, model.parameters())
        optimizer = optim.Adam(trainable_params, lr=0.001)

        train_losses = []
        train_accuracies = []

        for epoch in range(num_epochs):
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            loss, acc = train_one_epoch(model, train_dataloader, criterion, optimizer, device)
            train_losses.append(loss)
            train_accuracies.append(acc)
            print(f"\tLoss: {loss:.4f}, Accuracy: {acc:.2%}")

        # Avaliação
        print("Avaliação no conjunto de teste")

        model.eval()
        test_correct = 0
        test_total = 0
        all_labels = []
        all_predictions = []

        with torch.no_grad():
            for images, labels in test_dataloader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                _, predicted = torch.max(outputs, 1)

                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()

                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predicted.cpu().numpy())

        test_accuracy = test_correct / test_total

        print(f"\nAccuracy no teste: {test_accuracy:.2%}")
        print(f"Corretos: {test_correct}/{test_total}")

        cm = confusion_matrix(all_labels, all_predictions)
        print(f"\nMatriz de confusão (diagonal): {cm.diagonal()}")

        print("\nRelatório de classificação:")
        print(classification_report(all_labels, all_predictions,
                                   target_names=sorted(df['dx'].unique()),
                                   digits=4))

        results = {
            'model': model_name,
            'epochs': num_epochs,
            'test_accuracy': test_accuracy,
            'train_loss_final': train_losses[-1],
            'train_acc_final': train_accuracies[-1],
            'test_correct': test_correct,
            'test_total': test_total,
            'confusion_matrix': cm.tolist()
        }

        all_results.append(results)

        # Salvar Modelo
        save_dir = "models"
        os.makedirs(save_dir, exist_ok=True)
        save_path = f"{save_dir}/{model_name.replace(' ', '_')}_{num_epochs}epochs.pth"

        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'epochs': num_epochs,
            'accuracy': test_accuracy,
            'results': results
        }, save_path)
        print(f"\nModelo salvo em: {save_path}")

        print(f"Fim: {model_name} - {num_epochs} epochs")


Modelo: VGG16
Treinando VGG16 por 3 epochs


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 72.6MB/s]



Epoch 1/3
Batch 0/251 - Loss: 1.92
Batch 50/251 - Loss: 1.79
Batch 100/251 - Loss: 0.99
Batch 150/251 - Loss: 0.74
Batch 200/251 - Loss: 0.76
Batch 250/251 - Loss: 1.10
	Loss: 1.1560, Accuracy: 65.03%

Epoch 2/3
Batch 0/251 - Loss: 1.06
Batch 50/251 - Loss: 0.95
Batch 100/251 - Loss: 1.03
Batch 150/251 - Loss: 1.29
Batch 200/251 - Loss: 1.02
Batch 250/251 - Loss: 0.94
	Loss: 1.0896, Accuracy: 66.41%

Epoch 3/3
Batch 0/251 - Loss: 0.83
Batch 50/251 - Loss: 1.34
Batch 100/251 - Loss: 2.08
Batch 150/251 - Loss: 1.32
Batch 200/251 - Loss: 0.98
Batch 250/251 - Loss: 0.76
	Loss: 1.0082, Accuracy: 68.29%
AVALIAÇÃO NO CONJUNTO DE TESTE

Accuracy no teste: 72.34%
Corretos: 1449/2003

Matriz de confusão (diagonal): [ 117 1222    0   49   16   32   13]

Relatório de classificação:
              precision    recall  f1-score   support

       akiec     0.4500    0.5318    0.4875       220
         bcc     0.8336    0.9113    0.8707      1341
         bkl     0.0000    0.0000    0.0000        23
 

...


Epoch 1/3
Batch 0/251 - Loss: 2.06
Batch 50/251 - Loss: 0.68
Batch 100/251 - Loss: 0.64
Batch 150/251 - Loss: 0.65
Batch 200/251 - Loss: 0.94
Batch 250/251 - Loss: 0.47
	Loss: 0.9170, Accuracy: 70.29%

Epoch 2/3
Batch 0/251 - Loss: 0.76
Batch 50/251 - Loss: 0.84
Batch 100/251 - Loss: 0.93
Batch 150/251 - Loss: 0.42
Batch 200/251 - Loss: 0.61
Batch 250/251 - Loss: 1.19
	Loss: 0.7513, Accuracy: 73.50%

Epoch 3/3
Batch 0/251 - Loss: 1.02
Batch 50/251 - Loss: 0.83
Batch 100/251 - Loss: 0.69
Batch 150/251 - Loss: 0.84
Batch 200/251 - Loss: 0.63
Batch 250/251 - Loss: 0.47
	Loss: 0.7155, Accuracy: 74.80%
AVALIAÇÃO NO CONJUNTO DE TESTE

Accuracy no teste: 74.74%
Corretos: 1497/2003

Matriz de confusão (diagonal): [ 119 1164    6  104   15   51   38]

Relatório de classificação:
              precision    recall  f1-score   support

       akiec     0.5535    0.5409    0.5471       220
         bcc     0.8865    0.8680    0.8772      1341
         bkl     0.6667    0.2609    0.3750        23
 

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B2_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Epoch 1/5
Batch 0/251 - Loss: 1.92
Batch 50/251 - Loss: 1.07
Batch 100/251 - Loss: 0.93
Batch 150/251 - Loss: 0.94
Batch 200/251 - Loss: 1.21
Batch 250/251 - Loss: 0.79
	Loss: 0.9159, Accuracy: 69.45%

Epoch 2/5
Batch 0/251 - Loss: 0.89
Batch 50/251 - Loss: 0.57
Batch 100/251 - Loss: 0.83
Batch 150/251 - Loss: 0.87
Batch 200/251 - Loss: 0.51
Batch 250/251 - Loss: 0.65
	Loss: 0.7534, Accuracy: 73.39%

Epoch 3/5
Batch 0/251 - Loss: 0.94
Batch 50/251 - Loss: 0.44
Batch 100/251 - Loss: 0.73
Batch 150/251 - Loss: 0.59
Batch 200/251 - Loss: 0.47
Batch 250/251 - Loss: 0.83
	Loss: 0.7049, Accuracy: 74.88%

Epoch 4/5
Batch 0/251 - Loss: 0.84
Batch 50/251 - Loss: 0.90
Batch 100/251 - Loss: 0.49
Batch 150/251 - Loss: 0.79
Batch 200/251 - Loss: 0.61
Batch 250/251 - Loss: 0.49
	Loss: 0.6930, Accuracy: 74.81%

Epoch 5/5
Batch 0/251 - Loss: 0.60
Batch 50/251 - Loss: 0.79
Batch 100/251 - Loss: 0.65
Batch 150/251 - Loss: 0.72
Batch 200/251 - Loss: 0.67
Batch 250/251 - Loss: 0.51
	Loss: 0.6779, Accurac

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B2_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Epoch 1/10
Batch 0/251 - Loss: 1.89
Batch 50/251 - Loss: 0.64
Batch 100/251 - Loss: 0.74
Batch 150/251 - Loss: 0.59
Batch 200/251 - Loss: 0.61
Batch 250/251 - Loss: 0.49
	Loss: 0.9094, Accuracy: 69.50%

Epoch 2/10
Batch 0/251 - Loss: 0.77
Batch 50/251 - Loss: 0.97
Batch 100/251 - Loss: 0.60
Batch 150/251 - Loss: 1.28


Resumo dos testes

In [ ]:
print("Resumo Todos Experimentos")

for i, result in enumerate(all_results):
    print(f"\n{i+1}. {result['model']} - {result['epochs']} epochs")
    print(f"\tAccuracy: {result['test_accuracy']:.2%}")
    print(f"\tLoss final: {result['train_loss_final']:.4f}")

if all_results:
    best_result = max(all_results, key=lambda x: x['test_accuracy'])
    print(f"Melhor Modelo: {best_result['model']} com {best_result['epochs']} epochs")
    print(f"Acuracia: {best_result['test_accuracy']:.2%}")

Resultados obtidos em uma das execuções:

Resumo Todos Experimentos

1. VGG16 - 3 epochs
        Accuracy: 69.30%
        Loss final: 1.0018

2. VGG16 - 5 epochs
        Accuracy: 72.14%
        Loss final: 0.9645

3. VGG16 - 10 epochs
        Accuracy: 73.69%
        Loss final: 0.8748

4. ResNet18 - 3 epochs
        Accuracy: 72.74%
        Loss final: 0.7054

5. ResNet18 - 5 epochs
        Accuracy: 74.04%
        Loss final: 0.6700

6. ResNet18 - 10 epochs
        Accuracy: 76.64%
        Loss final: 0.6349

7. ResNet50 - 3 epochs
        Accuracy: 74.74%
        Loss final: 0.7054

8. ResNet50 - 5 epochs
        Accuracy: 76.39%
        Loss final: 0.6764

9. ResNet50 - 10 epochs
        Accuracy: 76.78%
        Loss final: 0.6526

10. EfficientNet-B0 - 3 epochs
        Accuracy: 75.39%
        Loss final: 0.7043

11. EfficientNet-B0 - 5 epochs
        Accuracy: 75.99%
        Loss final: 0.6730

12. EfficientNet-B0 - 10 epochs
        Accuracy: 77.28%
        Loss final: 0.6286

13. EfficientNet-B2 - 3 epochs
        Accuracy: 74.89%
        Loss final: 0.7020

14. EfficientNet-B2 - 5 epochs
        Accuracy: 76.54%
        Loss final: 0.6723

15. EfficientNet-B2 - 10 epochs
        Accuracy: 76.59%
        Loss final: 0.6352

16. SwinTransformer - 3 epochs
        Accuracy: 74.69%
        Loss final: 0.7003

17. SwinTransformer - 5 epochs
        Accuracy: 77.08%
        Loss final: 0.6456

18. SwinTransformer - 10 epochs
        Accuracy: 77.83%
        Loss final: 0.5987

19. DenseNet121 - 3 epochs
        Accuracy: 76.49%
        Loss final: 0.6700

20. DenseNet121 - 5 epochs
        Accuracy: 77.53%
        Loss final: 0.6327

21. DenseNet121 - 10 epochs
        Accuracy: 78.13%
        Loss final: 0.5863

22. MobileNetV3 - 3 epochs
        Accuracy: 77.98%
        Loss final: 0.5942

23. MobileNetV3 - 5 epochs
        Accuracy: 78.58%
        Loss final: 0.5185

24. MobileNetV3 - 10 epochs
        Accuracy: 77.58%
        Loss final: 0.3954

25. ConvNeXt - 3 epochs
        Accuracy: 77.03%
        Loss final: 0.6394

26. ConvNeXt - 5 epochs
        Accuracy: 78.98%
        Loss final: 0.5923

27. ConvNeXt - 10 epochs
        Accuracy: 80.03%
        Loss final: 0.5286
Melhor Modelo: ConvNeXt com 10 epochs
Acuracia: 80.03%

Treino executado apenas na convNeXt, 50 epochs, early stop
Accuracy no teste: 80.97%
Corretos: 1217/1503

Relatório de classificação:
              precision    recall  f1-score   support

       akiec     0.7244    0.5576    0.6301       165
         bcc     0.8484    0.9682    0.9044      1006
         bkl     0.7692    0.5882    0.6667        17
          df     0.6935    0.2575    0.3755       167
         mel     1.0000    0.5909    0.7429        22
          nv     0.7077    0.5974    0.6479        77
        vasc     0.5200    0.7959    0.6290        49

    accuracy                         0.8097      1503
   macro avg     0.7519    0.6222    0.6566      1503
weighted avg     0.8010    0.8097    0.7883      1503

# Respostas Slides

1. De acordo com os resultados obtidos, a rede escolhida para avaliação foi:
-  ConvNeXt - 10 epochs, com uma acurária de aprox. **80**%, devido a sua melhor acurácia no periodo de tempo semelhante aos outros.

2. As adaptações realizadas, para testes e comparação de resultados, foram principalmente:
- Aplicação de dataloaders, para carregamento em batches dos dados.
- Transformações nas imagens para encaixar melhor nos modelos (Redimensionalização, Normalização, Vetorização...)
- Aplicação de vários modelos de redes diferentes para comparação (com parametros diferentes para cada rede).
- Variação na quantidade de epochs no treinamento (3, 5 e 10, quantidades para o colab suportar execução de todas as redes).

3. Ao executar apenas o modelo, com no máximo 50 epochs e early stop, a acurácia máxima obtida foi de aprox. **82**%, não se afastando muito do resultado obtido anteriormente, mas executando em 33 epochs (aprox. 20 epochs a mais por 2% de acurácia). No ponto de vista de tradeoff acurácia x tempo de execução, não há ganho signficativo ao rodar mais epochs para o modelo nesse determinado dataset.

